# BookLens Story Timeline

Interactive visualization of story events, relationships, and reveals across a knowledge graph.

In [41]:
import json
from neo4j import GraphDatabase
from IPython.display import HTML, display
import pandas as pd

In [42]:
# Neo4j configuration
NEO4J_URI      = "bolt://localhost:7687"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "neo4j"

# Query parameters
SERIES_ID      = "the-red-rising-saga"
TO_CHAPTER     = 15  # reader's current chapter index

# Test mode: use mock data instead of Neo4j
USE_MOCK = False

In [43]:
# Query functions

def fetch_chapters(session, series_id, to_chapter_index):
    """Fetch chapters in reading window."""
    query = """
    MATCH (c:Chapter {series_id: $series_id})
    WHERE c.chapter_index <= $to_chapter_index
    RETURN c.chapter_index AS chapter_index,
           c.name AS name,
           c.summary AS summary,
           c.book_id AS book_id
    ORDER BY c.chapter_index ASC
    """
    result = session.run(query, series_id=series_id, to_chapter_index=to_chapter_index)
    return [dict(record) for record in result]


def fetch_characters(session, series_id, to_chapter_index):
    """Fetch characters introduced so far."""
    query = """
    MATCH (char:Character {series_id: $series_id})
    WHERE char.first_chapter_index <= $to_chapter_index
    RETURN char.name AS name,
           char.aliases AS aliases,
           char.faction AS faction,
           char.role AS role,
           char.description AS description,
           char.first_chapter_index AS first_chapter_index
    ORDER BY char.first_chapter_index ASC
    """
    result = session.run(query, series_id=series_id, to_chapter_index=to_chapter_index)
    return [dict(record) for record in result]


def fetch_relationships(session, series_id, to_chapter_index):
    """Fetch relationships introduced so far."""
    query = """
    MATCH (a:Character {series_id: $series_id})-[r]->(b:Character {series_id: $series_id})
    WHERE r.first_chapter <= $to_chapter_index
      AND type(r) IN ['ALLY','ENEMY','FAMILY','ROMANCE','MENTOR','RIVAL','OTHER']
    RETURN a.name AS char_a,
           b.name AS char_b,
           type(r) AS rel_type,
           r.description AS description,
           r.first_chapter AS introduced_in
    """
    result = session.run(query, series_id=series_id, to_chapter_index=to_chapter_index)
    return [dict(record) for record in result]


def fetch_reveals(session, series_id, to_chapter_index):
    """Fetch identity reveals so far."""
    try:
        query = """
        MATCH (a:Character {series_id: $series_id})-[r:REVEALED_AS]->(b:Character {series_id: $series_id})
        WHERE r.reveal_chapter_index <= $to_chapter_index
        RETURN a.name AS from_name,
               b.name AS to_name,
               r.reveal_chapter_index AS chapter_index,
               r.context AS context
        """
        result = session.run(query, series_id=series_id, to_chapter_index=to_chapter_index)
        return [dict(record) for record in result]
    except Exception:
        # REVEALED_AS relationship not yet in database
        return []

In [44]:
def build_events(chapters, characters, relationships, reveals):
    """Build event list from Neo4j results."""
    events = []

    # One event per chapter (the chapter summary beat)
    for i, ch in enumerate(chapters):
        events.append({
            "id": f"ch_{ch['chapter_index']}",
            "chapter_index": ch['chapter_index'],
            "chapter_name": ch['name'],
            "label": ch['name'],
            "body": (ch['summary'][:280] + "…") if len(ch.get('summary', "")) > 300 else ch.get('summary', ""),
            "chars": [],
            "turning": False,
            "type": "chapter_summary",
            "t_raw": i * 0.3,  # Space chapters 0.3 apart instead of 1.0
            "above": True
        })

    # One event per identity reveal — always a turning point
    for rev in reveals:
        if rev.get('from_name') and rev.get('to_name'):
            events.append({
                "id": f"reveal_{rev['from_name']}_{rev['to_name']}",
                "chapter_index": rev['chapter_index'],
                "chapter_name": next((c['name'] for c in chapters if c['chapter_index']==rev['chapter_index']), ""),
                "label": f"{rev['from_name']} revealed",
                "body": rev.get('context') or f"{rev['from_name']} is revealed to be {rev['to_name']}.",
                "chars": [rev['from_name'], rev['to_name']],
                "turning": True,
                "type": "reveal",
                "t_raw": rev['chapter_index'] * 0.3 + 0.15,  # Offset within chapter space
                "above": True
            })

    # Sort by t_raw
    events.sort(key=lambda e: e['t_raw'])
    
    # Normalize t to [0.0, 1.0]
    max_t = max((e['t_raw'] for e in events), default=1)
    for e in events:
        e['t'] = round(e['t_raw'] / max_t, 4) if max_t > 0 else 0.0

    return events

In [45]:
def render_timeline(events, chapters, series_title=""):
    """Render interactive timeline visualization as HTML+JS."""
    
    # Compute chapter dividers
    ch_t = {}
    for ev in events:
        ci = ev['chapter_index']
        if ci not in ch_t:
            ch_t[ci] = ev['t']
    
    chapters_by_index = {c['chapter_index']: c for c in chapters}
    ch_dividers = [{
        "chapter_index": k,
        "name": chapters_by_index.get(k, {}).get('name', ''),
        "t": v
    } for k, v in sorted(ch_t.items())]
    
    events_json = json.dumps(events)
    chapters_json = json.dumps([{"chapter_index": c["chapter_index"], "name": c["name"]} for c in chapters])
    ch_dividers_json = json.dumps(ch_dividers)
    
    html = f"""
<div style="font-family: Georgia, serif; padding: 1.5rem 0;">
  <div style="margin-bottom: 1.5rem;">
    <h2 style="margin: 0 0 0.5rem 0; font-size: 24px; font-weight: 500;">{series_title}</h2>
    <p style="margin: 0; font-size: 13px; color: #999;">Story Timeline</p>
  </div>
  
  <div id="mm-wrap" style="width:100%; height:48px; position:relative; cursor:col-resize; border:1px solid #e0ded8; border-radius:2px; background:#fafaf8; box-sizing:border-box;">
    <canvas id="mm" style="display:block; width:100%; height:100%; box-sizing:border-box;"></canvas>
  </div>
  
  <div id="mc-wrap" style="width:100%; height:220px; position:relative; margin-top:6px; border:1px solid #e0ded8; border-radius:2px; background:#fafaf8; box-sizing:border-box;">
    <canvas id="mc" style="display:block; width:100%; height:100%; box-sizing:border-box;"></canvas>
  </div>
  
  <div id="digest" style="margin-top:16px; padding:12px; border-top:1px solid #e0ded8; min-height:80px;">
    <span style="font-size:13px; color:#b0ada4;">Hover any event to read its beat</span>
  </div>
</div>

<script>
(function() {{
  const EVENTS = {events_json};
  const CHAPTERS = {chapters_json};
  const CH_DIVIDERS = {ch_dividers_json};

  const NOISE = Array.from({{length: 256}}, () => Math.random() * 2 - 1);

  function smoothNoise(x) {{
    const i = Math.floor(x * 40) & 255;
    const f = x * 40 - Math.floor(x * 40);
    return NOISE[i] * (1 - f) + NOISE[(i + 1) & 255] * f;
  }}

  function waveY(t, midY, amp) {{
    return midY
      + Math.sin(t * Math.PI * 3.4) * amp * 0.65
      + Math.sin(t * Math.PI * 7.2 + 1.1) * amp * 0.22
      + smoothNoise(t) * amp * 0.16;
  }}

  function init() {{
    const mmCanvas = document.getElementById('mm');
    const mcCanvas = document.getElementById('mc');
    const mmWrap = document.getElementById('mm-wrap');
    const mcWrap = document.getElementById('mc-wrap');

    if (!mmCanvas || !mcCanvas) return;

    const mmCtx = mmCanvas.getContext('2d');
    const mcCtx = mcCanvas.getContext('2d');

    mmCanvas.width = mmWrap.clientWidth || 800;
    mmCanvas.height = mmWrap.clientHeight || 48;
    mcCanvas.width = mcWrap.clientWidth || 800;
    mcCanvas.height = mcWrap.clientHeight || 220;

    const MM_PAD = {{left: 20, right: 20, top: 8, bottom: 8}};
    const MM_WAVE_Y = mmCanvas.height / 2;
    const MM_AMP = 8;

    const MC_PAD = {{left: 40, right: 20, top: 20, bottom: 40}};
    const MC_WAVE_Y = (mcCanvas.height - MC_PAD.top - MC_PAD.bottom) / 2 + MC_PAD.top;
    const MC_AMP = 40;

    let winS = 0.0;
    let winE = 0.3;
    let dragMode = null;
    let dragStart = null;
    let hoveredEventId = null;

    function drawMinimap() {{
      mmCtx.clearRect(0, 0, mmCanvas.width, mmCanvas.height);
      
      const plotW = mmCanvas.width - MM_PAD.left - MM_PAD.right;
      const plotX = MM_PAD.left;
      const plotY0 = MM_WAVE_Y;
      
      mmCtx.strokeStyle = '#1a1a18';
      mmCtx.lineWidth = 1;
      mmCtx.beginPath();
      for (let px = 0; px < plotW; px++) {{
        const t = px / plotW;
        const y = waveY(t, plotY0, MM_AMP);
        if (px === 0) mmCtx.moveTo(plotX + px, y);
        else mmCtx.lineTo(plotX + px, y);
      }}
      mmCtx.stroke();
      
      for (const ev of EVENTS) {{
        const x = plotX + ev.t * plotW;
        const y = plotY0;
        mmCtx.fillStyle = ev.turning ? '#E24B4A' : '#1a1a18';
        mmCtx.beginPath();
        mmCtx.arc(x, y, ev.turning ? 3 : 2, 0, Math.PI * 2);
        mmCtx.fill();
      }}
      
      const rectX = plotX + winS * plotW;
      const rectW = (winE - winS) * plotW;
      mmCtx.fillStyle = 'rgba(0, 0, 0, 0.06)';
      mmCtx.fillRect(rectX, MM_PAD.top, rectW, mmCanvas.height - MM_PAD.top - MM_PAD.bottom);
      mmCtx.strokeStyle = '#1a1a18';
      mmCtx.lineWidth = 1;
      mmCtx.strokeRect(rectX, MM_PAD.top, rectW, mmCanvas.height - MM_PAD.top - MM_PAD.bottom);
      
      const handleH = (mmCanvas.height - MM_PAD.top - MM_PAD.bottom) * 0.6;
      const handleY = MM_PAD.top + (mmCanvas.height - MM_PAD.top - MM_PAD.bottom - handleH) / 2;
      mmCtx.fillStyle = '#1a1a18';
      mmCtx.fillRect(rectX - 3, handleY, 6, handleH);
      mmCtx.fillRect(rectX + rectW - 3, handleY, 6, handleH);
    }}

    function drawMainCanvas() {{
      mcCtx.clearRect(0, 0, mcCanvas.width, mcCanvas.height);
      
      const plotW = mcCanvas.width - MC_PAD.left - MC_PAD.right;
      const plotX = MC_PAD.left;
      const plotY0 = MC_WAVE_Y;
      
      mcCtx.strokeStyle = 'rgba(0, 0, 0, 0.1)';
      mcCtx.lineWidth = 0.5;
      mcCtx.setLineDash([3, 5]);
      for (const div of CH_DIVIDERS) {{
        if (div.t >= winS && div.t <= winE) {{
          const x = plotX + (div.t - winS) / (winE - winS) * plotW;
          mcCtx.beginPath();
          mcCtx.moveTo(x, MC_PAD.top);
          mcCtx.lineTo(x, mcCanvas.height - MC_PAD.bottom);
          mcCtx.stroke();
          
          mcCtx.fillStyle = '#999';
          mcCtx.font = '10px sans-serif';
          mcCtx.textAlign = 'center';
          mcCtx.fillText(div.name.substring(0, 15).toUpperCase(), x, mcCanvas.height - 8);
        }}
      }}
      mcCtx.setLineDash([]);
      
      mcCtx.strokeStyle = '#1a1a18';
      mcCtx.lineWidth = 1.5;
      mcCtx.beginPath();
      for (let px = 0; px <= plotW; px++) {{
        const tNorm = px / plotW;
        const t = winS + tNorm * (winE - winS);
        const y = waveY(t, plotY0, MC_AMP);
        if (px === 0) mcCtx.moveTo(plotX + px, y);
        else mcCtx.lineTo(plotX + px, y);
      }}
      mcCtx.stroke();
      
      const visibleEvents = EVENTS.filter(e => e.t >= winS && e.t <= winE);
      for (const ev of visibleEvents) {{
        const tNorm = (ev.t - winS) / (winE - winS);
        const x = plotX + tNorm * plotW;
        const y = waveY(ev.t, plotY0, MC_AMP);
        
        const isHovered = ev.id === hoveredEventId;
        const isTurning = ev.turning;
        
        mcCtx.fillStyle = isHovered ? (isTurning ? '#E24B4A' : '#1a1a18') : 'white';
        mcCtx.strokeStyle = isTurning ? '#E24B4A' : '#1a1a18';
        mcCtx.lineWidth = 1.5;
        mcCtx.beginPath();
        mcCtx.arc(x, y, 4.5, 0, Math.PI * 2);
        mcCtx.fill();
        mcCtx.stroke();
        
        if (isHovered) {{
          mcCtx.strokeStyle = isTurning ? 'rgba(226, 75, 74, 0.2)' : 'rgba(26, 26, 24, 0.2)';
          mcCtx.lineWidth = 1;
          mcCtx.beginPath();
          mcCtx.arc(x, y, 9.5, 0, Math.PI * 2);
          mcCtx.stroke();
        }}
        
        mcCtx.fillStyle = '#1a1a18';
        mcCtx.font = '10px Georgia, serif';
        mcCtx.textAlign = 'center';
        
        let label = ev.label;
        let lines = [label];
        if (label.length > 17) {{
          const mid = Math.ceil(label.length / 2);
          lines = [label.substring(0, mid), label.substring(mid)];
        }}
        
        const stalked = ev.above ? y - 28 : y + 28;
        for (let i = 0; i < lines.length; i++) {{
          const lineY = ev.above ? stalked - (lines.length - i - 1) * 11 : stalked + i * 11;
          mcCtx.fillText(lines[i], x, lineY);
        }}
        
        mcCtx.strokeStyle = '#1a1a18';
        mcCtx.lineWidth = 0.5;
        mcCtx.beginPath();
        mcCtx.moveTo(x, y + (ev.above ? -4.5 : 4.5));
        mcCtx.lineTo(x, y + (ev.above ? -28 : 28));
        mcCtx.stroke();
      }}
    }}

    function updateDigest(eventId) {{
      const ev = EVENTS.find(e => e.id === eventId);
      if (!ev) {{
        document.getElementById('digest').innerHTML = '<span style="font-size:13px; color:#b0ada4;">Hover any event to read its beat</span>';
        return;
      }}

      let meta = `Ch ${{ev.chapter_index}}`;
      if (ev.turning) meta += ' · <strong>turning point</strong>';

      let charChips = '';
      if (ev.chars && ev.chars.length > 0) {{
        charChips = '<div style="margin-top: 8px; display: flex; gap: 6px; flex-wrap: wrap;">';
        for (const char of ev.chars) {{
          charChips += `<span style="border:1px solid #ddd; border-radius:999px; padding:2px 9px; font-size:11px;">${{char}}</span>`;
        }}
        charChips += '</div>';
      }}

      const html = `
        <div class="d-meta" style="font-size:11px; color:#999; text-transform:uppercase; margin-bottom:4px;">${{meta}}</div>
        <div class="d-headline" style="font-family:Georgia; font-size:17px; font-weight:500; margin-bottom:8px;">${{ev.label}}</div>
        <div class="d-body" style="font-size:13px; color:#666; line-height:1.6;">${{ev.body}}</div>
        ${{charChips}}
      `;
      document.getElementById('digest').innerHTML = html;
    }}

    mmCanvas.addEventListener('mousedown', (e) => {{
      const rect = mmCanvas.getBoundingClientRect();
      const x = e.clientX - rect.left;
      const plotW = mmCanvas.width - MM_PAD.left - MM_PAD.right;
      const rectX = MM_PAD.left + winS * plotW;
      const rectW = (winE - winS) * plotW;
      
      if (x < rectX - 10 || x > rectX + rectW + 10) {{
        dragMode = 'pan';
      }} else if (Math.abs(x - rectX) < 10) {{
        dragMode = 'resizeLeft';
      }} else if (Math.abs(x - (rectX + rectW)) < 10) {{
        dragMode = 'resizeRight';
      }} else {{
        dragMode = 'drag';
      }}
      dragStart = {{x, winS, winE}};
    }});

    document.addEventListener('mousemove', (e) => {{
      if (dragMode && dragStart) {{
        const rect = mmCanvas.getBoundingClientRect();
        const x = e.clientX - rect.left;
        const plotW = mmCanvas.width - MM_PAD.left - MM_PAD.right;
        const dx = (x - dragStart.x) / plotW;
        const span = dragStart.winE - dragStart.winS;
        
        if (dragMode === 'drag') {{
          let newS = dragStart.winS + dx;
          let newE = dragStart.winE + dx;
          if (newS < 0) {{ newS = 0; newE = span; }}
          if (newE > 1) {{ newE = 1; newS = 1 - span; }}
          winS = newS;
          winE = newE;
        }} else if (dragMode === 'resizeLeft') {{
          let newS = dragStart.winS + dx;
          if (newS < 0) newS = 0;
          if (dragStart.winE - newS > 0.15) winS = newS;
        }} else if (dragMode === 'resizeRight') {{
          let newE = dragStart.winE + dx;
          if (newE > 1) newE = 1;
          if (newE - dragStart.winS > 0.15) winE = newE;
        }}
        
        drawMinimap();
        drawMainCanvas();
      }}
      
      const mcRect = mcCanvas.getBoundingClientRect();
      const mcX = e.clientX - mcRect.left;
      const mcY = e.clientY - mcRect.top;
      
      const plotW = mcCanvas.width - MC_PAD.left - MC_PAD.right;
      const plotX = MC_PAD.left;
      const plotY0 = MC_WAVE_Y;
      
      let nearest = null;
      let nearestDist = 40;
      
      for (const ev of EVENTS.filter(e => e.t >= winS && e.t <= winE)) {{
        const tNorm = (ev.t - winS) / (winE - winS);
        const x = plotX + tNorm * plotW;
        const y = waveY(ev.t, plotY0, MC_AMP);
        const dist = Math.hypot(mcX - x, mcY - y);
        
        if (dist < nearestDist) {{
          nearest = ev.id;
          nearestDist = dist;
        }}
      }}
      
      if (nearest !== hoveredEventId) {{
        hoveredEventId = nearest;
        updateDigest(nearest);
        drawMainCanvas();
      }}
    }});

    document.addEventListener('mouseup', () => {{
      dragMode = null;
      dragStart = null;
    }});

    drawMinimap();
    drawMainCanvas();
  }}

  // Run immediately if DOM is ready, otherwise wait
  if (document.readyState === 'loading') {{
    document.addEventListener('DOMContentLoaded', init);
  }} else {{
    init();
  }}
}})();
</script>
    """
    return html

In [46]:
# Mock data for testing without Neo4j
MOCK_CHAPTERS = [
    {"chapter_index": 0, "name": "Prologue", "summary": "The Red God demands blood. On a frozen battlefield, Darrow discovers the harsh reality of the solar system."},
    {"chapter_index": 1, "name": "Welcome to the Institute", "summary": "After years in the mines, Darrow enters the Institute. He must survive among the elite Gold children."},
    {"chapter_index": 2, "name": "The First Night", "summary": "Darrow makes his first allies in the barracks. Cassius emerges as a dangerous rival."},
    {"chapter_index": 3, "name": "Morning Blood", "summary": "The first trials begin. Darrow learns that the Institute is a brutal hierarchy where strength determines survival."},
    {"chapter_index": 4, "name": "Antonia's Game", "summary": "Darrow joins a small band. Antonia, a natural leader, tests his loyalty in a dangerous expedition."},
    {"chapter_index": 5, "name": "The Hunt", "summary": "Students hunt each other through ruins. Darrow uses cunning and strategy to outmaneuver stronger opponents."},
    {"chapter_index": 6, "name": "The Burning", "summary": "A camp burns. In the chaos, Darrow discovers the Red God's teachings are not what he believed."},
    {"chapter_index": 7, "name": "Turning Points", "summary": "Darrow's crew fractures. Cassius ascends, and the balance of power shifts in unexpected ways."},
    {"chapter_index": 8, "name": "White Crowns", "summary": "The elite class reveals itself. Darrow realizes the scale of corruption in the Society."},
    {"chapter_index": 9, "name": "The Black", "summary": "Darrow discovers a secret society within the Institute. A mentor figure offers him a path forward."},
    {"chapter_index": 10, "name": "Reds and Golds", "summary": "Darrow's past and present collide. He begins to understand his true role in a larger game."},
    {"chapter_index": 11, "name": "Final Games", "summary": "The Institute's trials culminate in a devastating test. Winners and losers are decided."},
    {"chapter_index": 12, "name": "Exams", "summary": "Final examinations determine each student's future. Darrow faces his greatest challenger yet."},
    {"chapter_index": 13, "name": "Bloodydamn", "summary": "A shocking betrayal. Everything Darrow thought he understood is overturned."},
    {"chapter_index": 14, "name": "Gold", "summary": "Darrow's graduation. He learns his true mission and the price of ambition."},
    {"chapter_index": 15, "name": "Red Rising", "summary": "Epilogue: The rebellion begins. Darrow is now among the elites—and the real war is just starting."}
]

MOCK_RELATIONSHIPS = [
    {"char_a": "Darrow", "char_b": "Cassius", "rel_type": "RIVAL", "description": "Cassius, a Natural born Gold, emerges as Darrow's greatest rival at the Institute.", "moments": [1, 3, 5], "introduced_in": 2},
    {"char_a": "Darrow", "char_b": "Antonia", "rel_type": "ALLY", "description": "Antonia becomes a natural leader among Darrow's crew, earning his respect through cunning.", "moments": [4, 6], "introduced_in": 4},
    {"char_a": "Darrow", "char_b": "Roque", "rel_type": "ALLY", "description": "Roque joins Darrow's inner circle, bringing intelligence and strategic insight.", "moments": [], "introduced_in": 3},
    {"char_a": "Antonia", "char_b": "Cassius", "rel_type": "ENEMY", "description": "Antonia and Cassius clash for dominance, creating tension within the Institute.", "moments": [], "introduced_in": 5},
    {"char_a": "Darrow", "char_b": "Pliny", "rel_type": "MENTOR", "description": "Pliny becomes an unexpected mentor, guiding Darrow through the Institute's dangers.", "moments": [], "introduced_in": 9}
]

MOCK_REVEALS = [
    {"from_name": "Darrow", "to_name": "Reaper of Lykos", "chapter_index": 11, "context": "Darrow's true potential as a warrior is revealed when he wins the final trials."},
    {"from_name": "Pliny", "to_name": "The Obsidian", "chapter_index": 13, "context": "Pliny is revealed to be more than just a mentor—a player in a larger game."}
]

In [47]:
# Run the pipeline

if USE_MOCK:
    chapters = MOCK_CHAPTERS
    relationships = MOCK_RELATIONSHIPS
    reveals = MOCK_REVEALS
    characters = []  # Not needed for mock timeline
else:
    try:
        driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
        with driver.session() as session:
            chapters = fetch_chapters(session, SERIES_ID, TO_CHAPTER)
            characters = fetch_characters(session, SERIES_ID, TO_CHAPTER)
            relationships = fetch_relationships(session, SERIES_ID, TO_CHAPTER)
            reveals = fetch_reveals(session, SERIES_ID, TO_CHAPTER)
        driver.close()
    except Exception as e:
        print(f"Neo4j connection failed: {e}")
        print("Falling back to mock data...")
        chapters = MOCK_CHAPTERS
        relationships = MOCK_RELATIONSHIPS
        reveals = MOCK_REVEALS
        characters = []

# Build events and render
events = build_events(chapters, characters, relationships, reveals)
html = render_timeline(events, chapters, series_title="Red Rising")
display(HTML(html))

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing relationship type is: REVEALED_AS)} {position: line: 2, column: 56, offset: 56} for query: '\n        MATCH (a:Character {series_id: $series_id})-[r:REVEALED_AS]->(b:Character {series_id: $series_id})\n        WHERE r.reveal_chapter_index <= $to_chapter_index\n        RETURN a.name AS from_name,\n               b.name AS to_name,\n               r.reveal_chapter_index AS chapter_index,\n               r.context AS context\n        '
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning

In [48]:
# Debug: Print events as table
events_df = pd.DataFrame([
    {
        "Type": e['type'],
        "Label": e['label'],
        "Chapter": e['chapter_index'],
        "t": f"{e['t']:.3f}",
        "Turning": "✓" if e['turning'] else "",
        "Chars": ", ".join(e['chars']) if e['chars'] else "-"
    }
    for e in events
])
print(events_df.to_string(index=False))

           Type             Label  Chapter     t Turning Chars
chapter_summary          Prologue        0 0.000             -
chapter_summary      1: Helldiver        1 0.067             -
chapter_summary   2: The Township        2 0.133             -
chapter_summary     3: The Laurel        3 0.200             -
chapter_summary       4: The Gift        4 0.267             -
chapter_summary 5: The First Song        5 0.333             -
chapter_summary     6: The Martyr        6 0.400             -
chapter_summary        7: Lazarus        7 0.467             -
chapter_summary         8: Dancer        8 0.533             -
chapter_summary        9: The Lie        9 0.600             -
chapter_summary    10: The Carver       10 0.667             -
chapter_summary           11: Mad       11 0.733             -
chapter_summary   12: The Carving       12 0.800             -
chapter_summary    13: Bad Things       13 0.867             -
chapter_summary    14: Andromedus       14 0.933       